# Laboratory 03 — Fourier series: periodic signals as harmonic sums

A tuning fork that rings at 300 Hz sings when a 100 Hz *square* wave drives the speaker beside
it, and stays silent when a 100 Hz *sine* wave does. This laboratory finds the 300 Hz inside the
square wave, measures how much of it there is, and then puts a resonator in front of the signal
to confirm the answer physically.

One step is designed to falsify something you probably believe. In Part 4 you will add more and
more terms to a square wave and watch the approximation fail to improve where it is worst.

## Model specification

| | |
|---|---|
| **System** | a real periodic signal of period T, represented by its complex coefficients c_n; observables are partial sums, harmonic amplitudes and phases |
| **Dynamics** | none — a change of representation; when the harmonics drive the module 02 oscillator, each drives it independently by linearity |
| **Boundary** | exact periodicity forwards and backwards forever; one period carries the whole signal |
| **Ensemble** | deterministic, except Part 7, whose noise is Gaussian, independent and seeded |
| **Ignored** | transients of every kind — driven results are steady state only — and convergence beyond piecewise-smooth signals |
| **Valid when** | the signal really is periodic and piecewise smooth, and partial sums near a jump are read with Gibbs in mind |
| **Failure modes** | treating the overshoot as removable error, applying the series to a signal that never repeats, summing harmonic responses through a nonlinearity |

All the physics lives in `wavelab.fourier` and `wavelab.oscillators`. Open them and read them —
nothing in this notebook re-implements any of it.

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# wavelab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy and matplotlib, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# wavelab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("wavelab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from wavelab import fourier, measurement, oscillators
from wavelab.validation import scaling_exponent, seed_study

OMEGA0 = 2.0  # rad/s, the fundamental of every periodic signal in this notebook
PERIOD = 2 * np.pi / OMEGA0
N_GRID = 4096
PHASE = 2 * np.pi * np.arange(N_GRID) / N_GRID  # one period, endpoint excluded
TIMES = PHASE / OMEGA0

# Every stochastic function takes its generator explicitly, so results are reproducible
# and no hidden global state can leak between cells.
rng = np.random.default_rng(2024)

print(f"fundamental omega0 = {OMEGA0} rad/s   period = {PERIOD:.4f} s")
print(f"odd harmonics at {[n * OMEGA0 for n in (1, 3, 5, 7)]} rad/s")

## Part 1 — The harmonic recipe, drawn

Module 00 gave you one rotating arrow per oscillation. A periodic signal needs one arrow per
*harmonic*, the nth turning n times faster than the first, laid tip to tail. The running tip
traces the waveform.

Beside it sits the spectrum: a stem for each harmonic, its height the amplitude. Waveform and
spectrum are two views of one object, and from here to the end of the course they will appear
side by side.

In [ ]:
def arrows(coefficients, phase):
    """Tip positions of the harmonic phasors laid tip-to-tail at one instant."""
    n_max = (coefficients.size - 1) // 2
    orders = np.arange(1, n_max + 1)
    steps = 2j * np.conj(coefficients[n_max + orders]) * np.exp(-1j * orders * phase)
    return np.concatenate([[0j], np.cumsum(steps)])


c_square = fourier.square_coefficients(9)
snapshot = PERIOD / 8
chain = arrows(c_square, OMEGA0 * snapshot)
orbit = np.array([arrows(c_square, p)[-1] for p in np.linspace(0, 2 * np.pi, 400)])

fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.8))
left.plot(orbit.real, orbit.imag, lw=0.9, color="0.7")
left.plot(chain.real, chain.imag, lw=1.3, marker="o", markersize=3)
left.plot([chain[-1].real], [chain[-1].imag], marker="o", markersize=7, color="crimson")
left.set_aspect("equal"), left.set_title("harmonic phasors, tip to tail")
left.set_xticks([]), left.set_yticks([])

orders = np.arange(1, 10)
amplitudes = np.abs(2 * c_square[9 + orders])
right.stem(orders, amplitudes)
right.set_xlabel("harmonic n"), right.set_ylabel("amplitude"), right.set_title("spectrum")
plt.tight_layout()
plt.show()

reference = fourier.partial_sum(c_square, OMEGA0, np.array([snapshot]))[0]
print(f"tip height at this instant = {chain[-1].imag:.5f}")
print(f"partial sum there          = {reference:.5f}")
print(f"arrow lengths 4/(pi n): {np.round(amplitudes[::2], 4)}  (n = 1, 3, 5, 7, 9)")

In [ ]:
import ipywidgets as widgets

PRESETS = {
    "square": fourier.square_coefficients,
    "triangle": fourier.triangle_coefficients,
    "sawtooth": fourier.sawtooth_coefficients,
    "pulse train (duty 0.25)": lambda n: fourier.pulse_train_coefficients(0.25, n),
}
EXACT = {
    "square": np.sign(np.sin(PHASE)),
    "triangle": (2 / np.pi) * np.arcsin(np.sin(PHASE)),
    "sawtooth": ((PHASE + np.pi) % (2 * np.pi) - np.pi) / np.pi,
    "pulse train (duty 0.25)": np.where(
        np.abs((PHASE + np.pi) % (2 * np.pi) - np.pi) <= 0.25 * np.pi, 1.0, 0.0
    ),
}


def mixer(preset="square", n_terms=5):
    coefficients = PRESETS[preset](n_terms)
    built = fourier.partial_sum(coefficients, OMEGA0, TIMES)

    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.4))
    left.plot(TIMES, EXACT[preset], lw=1.0, color="0.7", label="exact")
    left.plot(TIMES, built, lw=1.5, label=f"N = {n_terms}")
    left.set_xlabel("t (s)"), left.set_ylabel("f(t)"), left.legend(fontsize=8)

    orders = np.arange(1, n_terms + 1)
    right.stem(orders, np.abs(2 * coefficients[n_terms + orders]))
    right.set_xlabel("harmonic n"), right.set_ylabel("amplitude")
    plt.tight_layout()
    plt.show()

    error = np.sqrt(np.mean((built - EXACT[preset]) ** 2))
    print(f"{preset}: N = {n_terms}, rms error = {error:.4f}")


widgets.interact_manual(
    mixer,
    preset=widgets.Dropdown(options=list(PRESETS), value="square"),
    n_terms=widgets.IntSlider(min=1, max=31, step=2, value=5, description="N"),
);

## Part 2 — Predict

Commit to answers before running anything below.

1. Does a resonator tuned to 3 omega0 respond to a square-wave drive of repetition rate omega0?
   To a *sine* drive at omega0? What about a resonator tuned to 2 omega0?
2. How many terms does a square wave need before it is exactly a square wave?
3. Zoom in on the jump while adding terms. The wiggles narrow. Do they also get *shorter*?
4. Triangle or square: which is better reproduced by its first five harmonics, and what feature
   of the waveform decides?

**Your prediction:**

1. 
2. 
3. 
4. 

## Part 3 — Reconstruct the four presets, and measure the rate

Rebuild each waveform from its first N harmonics and measure how the error falls. The prediction
from the page is that smoothness sets the rate: a jump costs 1/n in the coefficients and gives
an error falling as N^(-1/2), while a corner costs 1/n^2 and gives N^(-3/2).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 6))
for ax, (name, builder) in zip(axes.ravel(), PRESETS.items(), strict=True):
    ax.plot(TIMES, EXACT[name], lw=1.0, color="0.7", label="exact")
    ax.plot(TIMES, fourier.partial_sum(builder(5), OMEGA0, TIMES), lw=1.0, label="N = 5")
    ax.plot(TIMES, fourier.partial_sum(builder(31), OMEGA0, TIMES), lw=1.2, label="N = 31")
    ax.set_title(name, fontsize=9)
    ax.set_xlabel("t (s)"), ax.set_ylabel("f(t)"), ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

for name, builder in PRESETS.items():
    worst = np.max(np.abs(fourier.partial_sum(builder(31), OMEGA0, TIMES) - EXACT[name]))
    print(f"{name:24s} worst pointwise error at N = 31: {worst:.4f}")

In [ ]:
ORDERS = [15, 31, 63, 127, 255]

plt.figure(figsize=(7, 4))
rates = {}
for name, builder in PRESETS.items():
    errors = [
        float(np.sqrt(np.mean((fourier.partial_sum(builder(n), OMEGA0, TIMES) - EXACT[name]) ** 2)))
        for n in ORDERS
    ]
    rates[name] = scaling_exponent(ORDERS, errors)
    plt.loglog(ORDERS, errors, marker="o", markersize=4, label=name)
plt.xlabel("N"), plt.ylabel("rms error"), plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("fitted exponent   theory   waveform")
for name, exponent in rates.items():
    theory = -1.5 if name == "triangle" else -0.5
    print(f"    {exponent:+.3f}        {theory:+.1f}    {name}")

## Part 4 — The overshoot that will not go away

Prediction 3, settled by measurement. Zoom on the jump and raise N. Watch two different things:
how *wide* the first wiggle is, and how *tall* it is.

In [ ]:
window = np.linspace(-0.08 * PERIOD, 0.08 * PERIOD, 4001)

plt.figure(figsize=(9, 4))
for n in (5, 9, 17, 31):
    plt.plot(window, fourier.partial_sum(fourier.square_coefficients(n), OMEGA0, window),
             lw=1.1, label=f"N = {n}")
plt.axhline(1.0, color="0.85", lw=0.8)
plt.axhline(1 + 2 * fourier.gibbs_overshoot(31), color="green", ls="--", lw=1.0,
            label="1 + 2 x 8.95%")
plt.ylim(0.6, 1.35)
plt.xlabel("t (s)"), plt.ylabel("f(t)"), plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
counts = [7, 15, 31, 63, 127, 255]
overshoots = [fourier.gibbs_overshoot(n) for n in counts]

plt.figure(figsize=(7, 3.6))
plt.semilogx(counts, [100 * o for o in overshoots], marker="o")
plt.axhline(8.9491, color="green", ls="--", lw=1.0)
plt.ylim(0, 12)
plt.xlabel("N"), plt.ylabel("overshoot (% of the jump)")
plt.tight_layout()
plt.show()

for n, o in zip(counts, overshoots, strict=True):
    print(f"N = {n:4d}   overshoot = {100 * o:.4f}% of the jump")
print("\nIt is settling on a constant, not heading for zero.")

## Part 5 — Hear the recipe

Four waveforms, all at the same pitch — the same fundamental, the same period, the same note.
Only the spectrum differs. Play them in order and listen for the brightness climbing as the
harmonics get stronger:

- **sine** — one harmonic, nothing else. Pure, and slightly dull.
- **triangle** — odd harmonics falling as 1/n^2. Soft, flute-like: fast decay sounds mellow.
- **square** — odd harmonics falling as 1/n. Hollow and woody, like a clarinet.
- **sawtooth** — *every* harmonic, falling as 1/n. Bright and buzzy, like a bowed string.

Your ear is running a spectrum analyser. Timbre *is* the column of numbers plotted in Part 3.

In [ ]:
RATE = 44_100
PITCH = 220.0  # A3; with 31 harmonics the highest partial is 6.8 kHz, far below Nyquist,
DURATION = 1.2  # so the band-limited partial sum IS the alias-free signal being studied.

samples_per_period = int(round(RATE / PITCH))
repeats = int(DURATION * PITCH)
one_period_phase = 2 * np.pi * np.arange(samples_per_period) / samples_per_period


def player(coefficients):
    """One audio player, RMS-matched so that timbre is the only thing that differs."""
    from IPython.display import Audio, display

    one = fourier.partial_sum(coefficients, 1.0, one_period_phase)
    tone = np.tile(one, repeats)
    tone = 0.2 * tone / np.sqrt(np.mean(tone**2))
    # A 5 ms raised-cosine fade at each end; without it the click at the edges is louder
    # than the difference we are listening for.
    fade = int(0.005 * RATE)
    ramp = 0.5 * (1 - np.cos(np.linspace(0, np.pi, fade)))
    tone[:fade] *= ramp
    tone[-fade:] *= ramp[::-1]
    # normalize=False matters: the default peak-normalises, which would undo the RMS match
    # above and make all four sound equally loud, destroying the comparison.
    display(Audio(tone, rate=RATE, normalize=False))


sine = np.zeros(3, dtype=complex)
sine[2] = 0.5  # a single harmonic: c_1 = 1/2 gives a unit-amplitude cosine

for name, coefficients in [
    ("sine", sine),
    ("triangle", fourier.triangle_coefficients(31)),
    ("square", fourier.square_coefficients(31)),
    ("sawtooth", fourier.sawtooth_coefficients(31)),
]:
    n_max = (coefficients.size - 1) // 2
    strengths = np.abs(2 * coefficients[n_max + np.arange(1, min(4, n_max + 1))])
    print(f"{name:9s} first harmonics: {np.round(strengths, 3)}")
    player(coefficients)

## Part 6 — A resonator reads the comb

Now the opening puzzle, quantitatively. Hold the square-wave drive fixed and sweep the
*resonator* across the drive's harmonics. Each harmonic is weighted by module 02's response
curve, so the sweep should peak wherever the resonator sits on a harmonic that exists — and
find nothing between them.

The prediction comes from the harmonic sum. The check comes from `simulate_forced`, which
integrates the equation of motion under a sampled square-wave force and knows nothing about
harmonics at all.

In [ ]:
MASS = 0.5
QUALITY = 30.0
HARMONICS = np.arange(1, 32, 2)  # the square wave's odd harmonics
FORCE_AMPLITUDES = 4 / (np.pi * HARMONICS)  # peak amplitude of each harmonic


def predicted_rms(resonator_omega):
    """Steady-state rms displacement, harmonic by harmonic through module 02's curve."""
    stiffness = MASS * resonator_omega**2
    damping = np.sqrt(MASS * stiffness) / QUALITY
    response = oscillators.steady_state_response(
        HARMONICS * OMEGA0, MASS, stiffness, damping, 1.0
    )
    return float(np.sqrt(0.5 * np.sum(np.abs(response * FORCE_AMPLITUDES) ** 2)))


sweep = np.linspace(0.5 * OMEGA0, 10.5 * OMEGA0, 600)
predicted = np.array([predicted_rms(w) for w in sweep])


def measured_rms(resonator_omega, periods=140):
    """The same number from a direct integration under a sampled square-wave force."""
    stiffness = MASS * resonator_omega**2
    damping = np.sqrt(MASS * stiffness) / QUALITY
    dt = 2 * np.pi / resonator_omega / 60
    steps = int(periods * PERIOD / dt)
    force = np.sign(np.sin(OMEGA0 * np.arange(steps + 1) * dt))
    run = oscillators.simulate_forced(MASS, stiffness, 0.0, 0.0, dt, force, damping=damping)
    settled = run.times > run.times[-1] * 0.6  # discard the transient
    return float(np.sqrt(np.mean(run.positions[settled] ** 2)))


checks = [3 * OMEGA0, 4 * OMEGA0, 5 * OMEGA0]
measured = [measured_rms(w) for w in checks]

plt.figure(figsize=(9, 4))
plt.semilogy(sweep / OMEGA0, predicted, lw=1.4, label="harmonic sum")
plt.semilogy(np.array(checks) / OMEGA0, measured, "o", markersize=8, color="crimson",
             label="direct integration")
for n in HARMONICS[:5]:
    plt.axvline(n, color="0.85", ls=":", lw=0.9)
plt.xlabel("resonator frequency / omega0"), plt.ylabel("rms displacement (m)")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

for w, m in zip(checks, measured, strict=True):
    p = predicted_rms(w)
    print(f"resonator at {w / OMEGA0:.0f} omega0: predicted {p:.5f}, integrated {m:.5f}, "
          f"ratio {m / p:.3f}")
print(f"\nresponse at 4 omega0 as a fraction of that at 3 omega0: "
      f"{predicted_rms(4 * OMEGA0) / predicted_rms(3 * OMEGA0):.4f}")

The resonator is a *selector*. It finds 3 omega0 inside a square wave of repetition rate omega0
because 3 omega0 is genuinely there, and it finds almost nothing at 4 omega0 because half-wave
symmetry deleted the even harmonics before the resonator ever saw them.

## Part 7 — Measurement, with an error bar

Extract the first three odd harmonics from a *noisy* sampled square wave. Report each as a value
with an uncertainty across seeds — never a bare number.

Twenty-four seeds here, not the eight module 02 used, and the reason is worth a sentence. The
quantity being measured is the *magnitude* of a complex coefficient, and with only eight records
the standard error is itself so uncertain that a three-sigma test stops meaning three sigma: we
saw five-sigma excursions that were luck rather than physics. When an error bar is estimated
from the same small sample it is describing, the estimate needs to be good enough to trust.

In [ ]:
NOISE_SIGMA = 0.05
SEEDS = 24  # see the note above on why this is not the 8 that module 02 used
clean_square = np.sign(np.sin(PHASE))


def extract(order):
    """A measurement of one harmonic's amplitude from a single noisy record."""

    def measure(generator):
        record = measurement.add_noise(clean_square, NOISE_SIGMA, generator)
        coefficients = fourier.fourier_coefficients(record, 5)
        return float(np.abs(2 * coefficients[5 + order]))

    return measure


print("harmonic   measured amplitude      truth    within 3 sigma?")
for order in (1, 3, 5):
    study = seed_study(extract(order), n_seeds=SEEDS)
    truth = 4 / (np.pi * order)
    print(f"    n = {order}    {study.mean:.4f} +/- {study.standard_error:.4f}   "
          f"{truth:.4f}   {study.agrees_with(truth, 3.0)}")

plt.figure(figsize=(9, 3.2))
plt.plot(TIMES, measurement.add_noise(clean_square, NOISE_SIGMA, np.random.default_rng(1)),
         lw=0.6, alpha=0.7, label="one noisy record")
plt.plot(TIMES, clean_square, lw=1.2, label="truth")
plt.xlabel("t (s)"), plt.ylabel("f(t)"), plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Part 8 — A real instrument

The same extraction, on a real recording. Record a sustained note on any instrument, export it
as a WAV, and drop it beside this notebook — in JupyterLite, drag it into the file browser on
the left; locally, put it in `notebooks/en/labs/`. Name it `my-note.wav`.

If no file is present the cell synthesises a stand-in so the notebook still runs, but the point
of the exercise is to compare two *real* instruments playing the same written note and see the
difference show up as a row of numbers.

In [ ]:
try:
    from scipy.io import wavfile

    rate, raw = wavfile.read("my-note.wav")
    if raw.ndim > 1:
        raw = raw.mean(axis=1)
    audio = raw.astype(float)
    audio /= np.max(np.abs(audio))
    source = "my-note.wav"
except (FileNotFoundError, OSError):
    rate = 44_100
    stand_in = fourier.partial_sum(fourier.sawtooth_coefficients(15), 1.0, one_period_phase)
    audio = np.tile(stand_in, 40)
    audio = measurement.add_noise(audio / np.max(np.abs(audio)), 0.01, rng)
    source = "a synthesised stand-in (drop my-note.wav here for the real thing)"

# Estimate the fundamental from the spectrum, then resample exactly one period onto a grid.
omega, spectrum_values = fourier.spectrum(audio - audio.mean(), 1 / rate)
positive = omega > 0
f0 = float(omega[positive][np.argmax(np.abs(spectrum_values[positive]))] / (2 * np.pi))
one_period = np.interp(
    np.arange(N_GRID) / N_GRID / f0, np.arange(audio.size) / rate, audio
)
coefficients = fourier.fourier_coefficients(one_period - one_period.mean(), 10)
strengths = np.abs(2 * coefficients[10 + np.arange(1, 11)])

plt.figure(figsize=(8, 3.4))
plt.bar(np.arange(1, 11), strengths / strengths[0])
plt.xlabel("harmonic n"), plt.ylabel("amplitude relative to fundamental")
plt.tight_layout()
plt.show()

print(f"source: {source}")
print(f"fundamental: {f0:.1f} Hz")
print(f"first five harmonic ratios: {np.round(strengths[:5] / strengths[0], 3)}")
print("That row of numbers is the instrument's timbre.")

## Part 9 — Automated checks

Each assertion below mirrors a claim on the module page, and each is also a test in the
project's suite. If one fails, the page is wrong — not the notebook.

In [ ]:
# 1. Closed forms against the coefficient integral. The triangle is continuous and lands near
#    machine precision; the waveforms with jumps stall at about one part in N, because a finite
#    grid cannot resolve a discontinuity.
triangle_error = np.max(np.abs(fourier.fourier_coefficients(EXACT["triangle"], 9)
                               - fourier.triangle_coefficients(9)))
square_error = np.max(np.abs(fourier.fourier_coefficients(EXACT["square"], 9)
                             - fourier.square_coefficients(9)))
assert triangle_error < 1e-6, triangle_error
assert square_error < 5 / N_GRID, square_error

# 2. Smoothness sets the convergence rate.
assert abs(rates["square"] - (-0.5)) < 0.05, rates["square"]
assert abs(rates["triangle"] - (-1.5)) < 0.05, rates["triangle"]

# 3. The Gibbs overshoot is constant in N, not shrinking. This is the falsifier.
assert abs(fourier.gibbs_overshoot(255) - 0.0895) < 0.001
assert abs(fourier.gibbs_overshoot(15) - fourier.gibbs_overshoot(255)) < 0.001

# 4. Parseval: the harmonics carry all the power.
band_limited = fourier.partial_sum(fourier.triangle_coefficients(9), OMEGA0, TIMES)
assert np.isclose(np.sum(np.abs(fourier.triangle_coefficients(9)) ** 2),
                  np.mean(band_limited**2), rtol=1e-10)

# 5. The comb: integration agrees with the harmonic sum on a harmonic, and parking the
#    resonator between harmonics drops the response by more than a factor of ten. It does not
#    fall to zero, and the reason is worth knowing: there is no even harmonic to respond to,
#    so what remains is the off-resonant tail of the third and fifth, not a fourth harmonic.
assert abs(measured[0] / predicted_rms(3 * OMEGA0) - 1) < 0.05
assert predicted_rms(4 * OMEGA0) / predicted_rms(3 * OMEGA0) < 0.1

# 6. The noisy extraction recovers every harmonic within its stated uncertainty.
for order in (1, 3, 5):
    assert seed_study(extract(order), n_seeds=SEEDS).agrees_with(4 / (np.pi * order), 3.0)

print("all checks passed")
print(f"square wave third harmonic: {4 / (3 * np.pi):.4f} — the answer to the opening puzzle")

## Part 10 — Explore it yourself

The integrator runs live in the panel below, so it uses a **Run** button (`interact_manual`)
rather than redrawing on every slider twitch: the browser kernel is too slow for that.

Three things worth trying:

1. Scramble the phases while leaving every amplitude alone. The waveform changes completely;
   the spectrum panel does not move at all. Then listen to it. (Module 04 asks what that means.)
2. Build the smoothest waveform you can from three harmonics, and the sharpest.
3. Set the pulse-train duty cycle very small and watch the spectrum flatten out — the comb
   under a widening envelope, which is where module 04 begins.

In [ ]:
def explore(a1=1.0, a3=0.333, a5=0.2, phase3=0.0, phase5=0.0, audible=False):
    coefficients = np.zeros(11, dtype=complex)
    for order, amplitude, phase in [(1, a1, 0.0), (3, a3, phase3), (5, a5, phase5)]:
        # c_n and c_-n are conjugates; the harmonic's phasor is 2 c_n*.
        coefficients[5 + order] = 0.5 * amplitude * np.exp(-1j * phase)
        coefficients[5 - order] = np.conj(coefficients[5 + order])

    built = fourier.partial_sum(coefficients, OMEGA0, TIMES)
    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.4))
    left.plot(TIMES, built, lw=1.5)
    left.set_xlabel("t (s)"), left.set_ylabel("f(t)"), left.set_ylim(-2.2, 2.2)
    right.stem([1, 3, 5], [a1, a3, a5])
    right.set_xlabel("harmonic n"), right.set_ylabel("amplitude"), right.set_ylim(0, 1.2)
    plt.tight_layout()
    plt.show()

    print(f"power in each harmonic: {np.round(0.5 * np.array([a1, a3, a5]) ** 2, 4)}")
    if audible:
        player(coefficients)


widgets.interact_manual(
    explore,
    a1=widgets.FloatSlider(min=0, max=1, step=0.02, value=1.0, description="|c1|"),
    a3=widgets.FloatSlider(min=0, max=1, step=0.02, value=0.333, description="|c3|"),
    a5=widgets.FloatSlider(min=0, max=1, step=0.02, value=0.2, description="|c5|"),
    phase3=widgets.FloatSlider(min=-np.pi, max=np.pi, step=0.1, value=0.0, description="arg c3"),
    phase5=widgets.FloatSlider(min=-np.pi, max=np.pi, step=0.1, value=0.0, description="arg c5"),
    audible=widgets.Checkbox(value=False, description="play it"),
);

## Check your understanding

Six questions on the module's objectives.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "03-fourier-series.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## Before you leave

1. Which of your four predictions fell, and what was the flawed model behind it?
2. "The square wave contains 300 Hz" — physical statement or mathematical bookkeeping? Defend
   your answer using only the Part 6 data.
3. Your Part 7 result has an error bar. Name what dominates it, and say how you would halve it.

**Your answers:**

1. 
2. 
3. 